# Market Basket Analysis con FP-Growth — Instacart

## Objetivo

Vamos a construir un **sistema de recomendación de productos a partir de un carrito de compras**, usando un enfoque clásico de minería de datos llamado **reglas de asociación** (association rules).

La pregunta que este modelo responde es: *"Si un cliente ya tiene los productos A y B en el carrito, ¿qué otro producto C es probable que también quiera?"*

## ¿Qué es una regla de asociación?

Una regla de asociación tiene la forma:

$$\text{antecedents} \rightarrow \text{consequents}$$

Por ejemplo: `{pan, manteca} → {mermelada}`. Esto significa: *"las órdenes que contienen pan y manteca, frecuentemente también contienen mermelada"*.

- **antecedents**: el conjunto de productos que ya están en el carrito (la "condición").
- **consequents**: el conjunto de productos que se recomienda agregar (la "consecuencia").

Estas reglas no se inventan a mano: se **descubren automáticamente** contando qué combinaciones de productos aparecen juntas con mucha frecuencia en el historial de órdenes. Ese descubrimiento se hace en dos pasos:

1. **Minar itemsets frecuentes**: encontrar qué combinaciones de productos aparecen juntas al menos en un cierto porcentaje de las órdenes. Acá es donde usamos el algoritmo **FP-Growth** (más abajo explicamos por qué).
2. **Generar reglas** a partir de esos itemsets, y quedarnos solo con las que son estadísticamente interesantes.

## Las métricas: support, confidence y lift

Estas son las tres métricas que vas a ver en cada regla. Vamos a definirlas con un ejemplo numérico chiquito para que quede clarísimo.

Supongamos que tenemos **100 órdenes** en total, y de ellas:
- 40 órdenes contienen **pan**.
- 30 órdenes contienen **manteca**.
- 20 órdenes contienen **pan Y manteca** juntos.

### Support — ¿qué tan frecuente es esta combinación?

$$\text{support}(A \rightarrow B) = \frac{\text{órdenes que contienen A y B}}{\text{total de órdenes}}$$

En nuestro ejemplo: `support(pan → manteca) = 20 / 100 = 0.20` (20%).

El support mide **popularidad**: qué tan seguido aparece esa combinación en el total de los datos. Un support muy bajo significa que es un patrón raro (poca evidencia estadística); un support alto significa que es un patrón común.

### Confidence — de los que compraron A, ¿cuántos compraron también B?

$$\text{confidence}(A \rightarrow B) = \frac{\text{support}(A \rightarrow B)}{\text{support}(A)} = \frac{\text{órdenes con A y B}}{\text{órdenes con A}}$$

En nuestro ejemplo: `confidence(pan → manteca) = 20 / 40 = 0.50` (50%).

Esto quiere decir: *"de las órdenes que tienen pan, el 50% también tiene manteca"*. La confidence mide **qué tan confiable/fiable es la regla** — es básicamente una probabilidad condicional P(manteca | pan).

**Ojo con un problema clásico de la confidence**: si manteca fuera un producto que TODO el mundo compra (por ejemplo, está en el 90% de las órdenes), entonces la confidence de `pan → manteca` sería alta simplemente porque manteca es súper popular, no porque haya una relación real con el pan. Para corregir esto existe el lift.

### Lift — ¿la relación es real o es pura casualidad estadística?

$$\text{lift}(A \rightarrow B) = \frac{\text{confidence}(A \rightarrow B)}{\text{support}(B)}$$

Sigamos el ejemplo: `support(manteca) = 30/100 = 0.30`. Entonces `lift(pan → manteca) = 0.50 / 0.30 = 1.67`.

El lift compara la confidence observada contra lo que esperaríamos **si pan y manteca fueran completamente independientes** (es decir, si comprar pan no tuviera ninguna relación con comprar manteca). Interpretación:

- **lift = 1**: A y B son independientes. No hay ninguna relación real; la regla no sirve para recomendar.
- **lift > 1**: A y B aparecen juntos **más** de lo que el azar predeciría. Relación positiva — esta es la que nos interesa para recomendar. En el ejemplo, lift=1.67 significa que comprar manteca es 1.67 veces más probable si ya compraste pan, comparado con un cliente al azar.
- **lift < 1**: A y B aparecen juntos **menos** de lo esperado (se "repelen", son sustitutos). Por ejemplo, dos marcas competidoras de la misma leche.

**Por eso el lift es la métrica más importante para recomendar**: no se deja engañar por productos populares y mide relación real, no solo co-ocurrencia.

### Otras métricas que vas a ver en la tabla completa (no las usamos para filtrar, pero es bueno saber qué son)

- **leverage**: `support(A→B) - support(A)*support(B)`. Similar al lift pero en escala absoluta en vez de razón; 0 significa independencia.
- **conviction**: mide qué tan seguido la regla "falla" (aparece A sin B) comparado con lo esperado al azar. Valores altos = regla muy confiable.

## ¿Por qué FP-Growth y no Apriori?

Apriori y FP-Growth, con el mismo `min_support`, encuentran **exactamente los mismos itemsets frecuentes** — el resultado matemático es idéntico. La diferencia está en **cómo llegan** a ese resultado:

- **Apriori** genera candidatos de a poco (pares, después tríos, después cuádruples...) y cuenta cada uno contra todo el dataset. Con miles de productos, la cantidad de combinaciones candidatas explota combinatoriamente.
- **FP-Growth** construye una sola vez un árbol compacto (**FP-tree**) que resume todas las transacciones, y mina los patrones frecuentes recorriendo ese árbol, sin generar candidatos explícitos. Escala mucho mejor cuando hay muchos productos y muchas transacciones, como en nuestro caso (49.677 productos, 3,2 millones de órdenes).

Por eso en este notebook usamos **únicamente FP-Growth**.

## 1. Imports y configuración

In [2]:
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import association_rules, fpgrowth

### Parámetros del modelo

Estos son los parámetros que gobiernan todo el análisis. Jugá con ellos para ver cómo cambian los resultados:

- **`MIN_SUPPORT`**: soporte mínimo para que un itemset se considere "frecuente". Es el parámetro más importante de todos: bajarlo genera muchísimas más combinaciones a explorar (más reglas, pero más tiempo y memoria); subirlo deja solo los patrones más masivos.
- **`MIN_CONFIDENCE`**: confianza mínima para que una regla se considere lo bastante fiable como para generarse.
- **`MIN_LIFT`**: lift mínimo para quedarnos con la regla en la salida final (descartamos lift ≤ 1, que no aportan valor real de recomendación).
- **`MAX_CONSEQUENT_SUPPORT`**: tope de popularidad para lo que el modelo puede *recomendar* (no afecta la tabla de reglas, solo la función `recommend_from_cart` — más adelante, en la sección 7.5, explicamos en detalle por qué hace falta y qué trade-off implica).

**Nota sobre memoria**: con `MIN_SUPPORT = 0.001` (0.1%), sobre las 3.214.874 órdenes del dataset completo, sobreviven ~1.770 productos con soporte individual suficiente (vs. ~840 con 0.002), y el paso de FP-Growth pasa de ~50s a un par de minutos. Si bajás aún más `MIN_SUPPORT`, más productos entran en juego y el paso de FP-Growth necesita más RAM y tiempo (más abajo explicamos por qué, con la cuenta exacta).

**Trade-off importante al bajar estos umbrales**: vas a obtener *muchas más* reglas con lift > 1, pero ojo — cuanto más bajo el `min_support`, más reglas quedan sostenidas por muy pocas órdenes (soporte bajo = poca evidencia estadística). Podés terminar con lifts altísimos que en realidad son ruido, no patrones reales. Más reglas no es sinónimo de mejor modelo; es un trade-off entre cobertura (menos carritos sin recomendación) y confiabilidad estadística de cada regla individual.

In [3]:
MIN_SUPPORT = 0.001      # 0.1% de las ordenes: aparece en al menos ~3215 ordenes
MIN_CONFIDENCE = 0.1     # al menos 10% de fiabilidad condicional
MIN_LIFT = 1.0           # descartamos reglas con lift <= 1 (sin valor real de recomendacion)
MAX_CONSEQUENT_SUPPORT = 0.10  # no recomendamos productos que ya esten en mas del 10% de las ordenes
RANDOM_STATE = 42

DATA_DIR = '../data/raw/instacart'
OUTPUT_PATH = '../data/processed/association_rules.csv'

## 2. Carga de datos

Usamos `order_products__prior.csv`: cada fila es *"este producto estuvo en esta orden"*. Es el historial de compras que vamos a minar.

Cargamos solo las columnas que necesitamos (`order_id`, `product_id`) y forzamos `dtype int32` en vez del `int64` por defecto de pandas — esto reduce a la mitad la memoria que ocupa el DataFrame (el dataset tiene 32,4 millones de filas, así que cada byte cuenta).

También cargamos `products.csv` para poder traducir `product_id → product_name` más adelante y que las reglas se puedan leer como humanos, no solo como números.

In [4]:
t0 = time.time()
order_products = pd.read_csv(
    f'{DATA_DIR}/order_products__prior.csv',
    usecols=['order_id', 'product_id'],
    dtype={'order_id': 'int32', 'product_id': 'int32'},
)
products = pd.read_csv(f'{DATA_DIR}/products.csv')
id_to_name = dict(zip(products['product_id'], products['product_name']))

print(f'order_products: {order_products.shape[0]:,} filas, {order_products.shape[1]} columnas')
print(f'ordenes unicas: {order_products["order_id"].nunique():,}')
print(f'productos unicos: {order_products["product_id"].nunique():,}')
print(f'tiempo de carga: {time.time() - t0:.1f}s')

order_products: 32,434,489 filas, 2 columnas
ordenes unicas: 3,214,874
productos unicos: 49,677
tiempo de carga: 5.6s


## 3. Codificación one-hot de las transacciones (la "canasta")

`mlxtend` necesita los datos en formato **one-hot**: una tabla donde cada **fila es una orden** y cada **columna es un producto**, con `True`/`False` según si ese producto estuvo en esa orden. A esto se lo llama la matriz de transacciones o "canasta" (basket).

**¿Por qué tiene que ser sparse (dispersa) y no una tabla común (densa)?** Porque el tamaño de la tabla densa sería `n_órdenes × n_productos`:

$$3.214.874 \text{ ordenes} \times 49.677 \text{ productos} \approx 160.000 \text{ millones de celdas}$$

Eso son ~160 GB solo para guardar `True`/`False` — inviable en cualquier compu. Pero la matriz real es **extremadamente dispersa**: cada orden tiene en promedio ~10 productos de los 49.677 posibles, así que más del 99.9% de las celdas son `False`. Una **matriz sparse** solo guarda las posiciones que son `True` (los ~32,4 millones de "presente"), no las millones de `False` — ahí es donde está el ahorro real de memoria.

En vez de armar listas de Python por orden (`groupby().apply(list)`, que es el camino típico con `TransactionEncoder` pero es lento sobre 3,2 millones de órdenes), construimos la matriz sparse directamente: convertimos `order_id` y `product_id` a **códigos categóricos** (enteros consecutivos) y los usamos como coordenadas fila/columna de una `scipy.sparse.csr_matrix`. Es matemáticamente el mismo one-hot encoding, mucho más rápido de construir a esta escala.

In [5]:
t0 = time.time()
order_cat = order_products['order_id'].astype('category')
product_cat = order_products['product_id'].astype('category')

row_idx = order_cat.cat.codes.to_numpy()          # que fila (orden) es cada linea
col_idx = product_cat.cat.codes.to_numpy()         # que columna (producto) es cada linea
n_orders = len(order_cat.cat.categories)
all_product_ids = product_cat.cat.categories.to_numpy()  # product_id real detras de cada columna

values = np.ones(len(order_products), dtype=bool)
basket_matrix = csr_matrix(
    (values, (row_idx, col_idx)),
    shape=(n_orders, len(all_product_ids)),
    dtype=bool,
)

print(f'matriz sparse: {basket_matrix.shape[0]:,} ordenes x {basket_matrix.shape[1]:,} productos')
print(f'valores True (presente): {basket_matrix.nnz:,} ({basket_matrix.nnz / basket_matrix.shape[0]:.1f} productos por orden en promedio)')
densidad = basket_matrix.nnz / (basket_matrix.shape[0] * basket_matrix.shape[1])
print(f'densidad: {densidad*100:.3f}% de las celdas son True')
print(f'tiempo: {time.time() - t0:.1f}s')

matriz sparse: 3,214,874 ordenes x 49,677 productos
valores True (presente): 32,434,489 (10.1 productos por orden en promedio)
densidad: 0.020% de las celdas son True
tiempo: 1.3s


## 4. Filtrar productos poco frecuentes antes de minar (paso obligatorio)

Acá hay un detalle técnico importante que vale la pena entender, porque no es solo una optimización: es un **requisito real** para que el algoritmo corra sin quedarse sin memoria.

**El problema**: la implementación de FP-Growth que usamos (`mlxtend` 0.25) necesita calcular el soporte de cada producto individual antes de construir el árbol, y para eso internamente **densifica** la matriz completa (la convierte de sparse a densa) en ese paso puntual. Si le pasamos los 49.677 productos completos, esa conversión intenta reservar los ~160 GB que mencionamos arriba y explota (`MemoryError`). Lo comprobamos directamente al intentarlo.

**La solución (y es una solución *correcta*, no un parche)**: antes de minar, descartamos los productos cuyo soporte individual ya es menor a `MIN_SUPPORT`. Esto se apoya en una propiedad matemática de los itemsets frecuentes llamada **anti-monotonicidad** (o *downward closure*): **si un producto no llega al soporte mínimo por sí solo, ningún itemset que lo contenga puede llegar al soporte mínimo tampoco** (agregar más productos a una combinación nunca puede aumentar su soporte, solo mantenerlo igual o bajarlo). Por lo tanto, es matemáticamente seguro eliminar esos productos de entrada: nunca iban a poder formar parte de una regla frecuente.

Este filtro reduce drásticamente la cantidad de columnas (de 49.677 a unos cientos, según el `MIN_SUPPORT` elegido), lo cual también resuelve el problema de memoria: la conversión a denso ahora es sobre una matriz mucho más chica.

**Cuenta de memoria** (para que puedas ajustar `MIN_SUPPORT` sabiendo el costo): el pico de RAM durante ese paso interno es aproximadamente `n_ordenes × n_productos_sobrevivientes` bytes (1 byte por celda booleana). Con `MIN_SUPPORT=0.002` sobreviven ~800 productos → `3.214.874 × 800 ≈ 2,6 GB` de pico. Si bajás `MIN_SUPPORT`, van a sobrevivir más productos y ese pico crece proporcionalmente.

In [6]:
item_support = np.asarray(basket_matrix.sum(axis=0)).ravel() / n_orders
keep_mask = item_support >= MIN_SUPPORT
kept_product_ids = all_product_ids[keep_mask]
basket_matrix_reduced = basket_matrix[:, keep_mask]

peak_ram_gb = n_orders * len(kept_product_ids) / 1e9
print(f'productos que sobreviven al filtro: {len(kept_product_ids):,} de {len(all_product_ids):,}')
print(f'pico de RAM estimado en el paso de FP-Growth: ~{peak_ram_gb:.2f} GB')

# mlxtend exige nombres de columna string (no puede haber columnas enteras en formato sparse)
basket = pd.DataFrame.sparse.from_spmatrix(
    basket_matrix_reduced,
    columns=[str(p) for p in kept_product_ids],
)
basket.shape

productos que sobreviven al filtro: 1,772 de 49,677
pico de RAM estimado en el paso de FP-Growth: ~5.70 GB


(3214874, 1772)

## 5. Minar itemsets frecuentes con FP-Growth

`fpgrowth` recorre la matriz de transacciones y devuelve todas las combinaciones de productos (`itemsets`) cuyo `support` es mayor o igual a `MIN_SUPPORT`. Internamente arma el FP-tree que mencionamos en la introducción.

In [7]:
t0 = time.time()
frequent_itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
elapsed = time.time() - t0

print(f'itemsets frecuentes encontrados: {len(frequent_itemsets):,}')
print(f'tiempo de FP-Growth: {elapsed:.1f}s')
frequent_itemsets.sort_values('support', ascending=False).head(10)

itemsets frecuentes encontrados: 3,883
tiempo de FP-Growth: 87.4s


,support,itemsets
40,0.146993,frozenset({24852})
15,0.118030,frozenset({13176})
41,0.082331,frozenset({21137})
4,0.075251,frozenset({21903})
16,0.066436,frozenset({47209})
42,0.054999,frozenset({47766})
141,0.047485,frozenset({47626})
192,0.044466,frozenset({16797})
257,0.043743,frozenset({26209})
61,0.042896,frozenset({27845})


## 6. Generar las reglas de asociación

`association_rules` toma los itemsets frecuentes y arma todas las reglas posibles `antecedents → consequents` a partir de sus subconjuntos, calculando `confidence` y `lift` para cada una. Filtramos por `MIN_CONFIDENCE` al generarlas, y después por `MIN_LIFT` (nos quedamos solo con relaciones reales, no casualidades).

El resultado final tiene exactamente las columnas que buscamos: `['antecedents', 'consequents', 'support', 'confidence', 'lift']`, ordenado por `lift` descendente (las relaciones más fuertes primero).

In [8]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules[rules['lift'] >= MIN_LIFT].copy()

# de vuelta a product_id como entero (mlxtend los devuelve como string porque asi los mandamos)
rules['antecedents'] = rules['antecedents'].apply(lambda s: frozenset(int(i) for i in s))
rules['consequents'] = rules['consequents'].apply(lambda s: frozenset(int(i) for i in s))

rules = (
    rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .sort_values('lift', ascending=False)
    .reset_index(drop=True)
)

print(f'reglas generadas: {len(rules):,}')
rules.head(10)

reglas generadas: 1,803


,antecedents,consequents,support,confidence,lift
0,frozenset({13263}),frozenset({28465}),0.001221,0.453129,75.647888
1,frozenset({28465}),frozenset({13263}),0.001221,0.203822,75.647888
2,frozenset({47977}),frozenset({36316}),0.001037,0.223406,75.634068
3,frozenset({36316}),frozenset({47977}),0.001037,0.351201,75.634068
4,frozenset({28465}),frozenset({36865}),0.002247,0.375136,73.640837
5,frozenset({36865}),frozenset({28465}),0.002247,0.441106,73.640837
6,frozenset({4957}),"frozenset({33754, 33787})",0.001161,0.179834,72.141821
7,"frozenset({33754, 33787})",frozenset({4957}),0.001161,0.465810,72.141821
8,frozenset({8309}),frozenset({28465}),0.001418,0.427553,71.378186
9,frozenset({28465}),frozenset({8309}),0.001418,0.236745,71.378186


### Leyendo una regla real

Traduzcamos a lenguaje humano las 3 reglas con mayor lift, usando los mismos números de la tabla:

In [9]:
for _, r in rules.head(3).iterrows():
    ante = ', '.join(id_to_name[p] for p in r['antecedents'])
    cons = ', '.join(id_to_name[p] for p in r['consequents'])
    print(f"Si el carrito tiene [{ante}]:")
    print(f"  -> el {r['confidence']*100:.1f}% de esas ordenes tambien llevo [{cons}]")
    print(f"  -> eso es {r['lift']:.1f}x mas probable que si fuera puro azar (lift={r['lift']:.2f})")
    print(f"  -> esta combinacion aparece en el {r['support']*100:.2f}% de TODAS las ordenes (support={r['support']:.4f})")
    print()

Si el carrito tiene [Non Fat Acai & Mixed Berries Yogurt]:
  -> el 45.3% de esas ordenes tambien llevo [Icelandic Style Skyr Blueberry Non-fat Yogurt]
  -> eso es 75.6x mas probable que si fuera puro azar (lift=75.65)
  -> esta combinacion aparece en el 0.12% de TODAS las ordenes (support=0.0012)

Si el carrito tiene [Icelandic Style Skyr Blueberry Non-fat Yogurt]:
  -> el 20.4% de esas ordenes tambien llevo [Non Fat Acai & Mixed Berries Yogurt]
  -> eso es 75.6x mas probable que si fuera puro azar (lift=75.65)
  -> esta combinacion aparece en el 0.12% de TODAS las ordenes (support=0.0012)

Si el carrito tiene [Grapefruit Sparkling Water]:
  -> el 22.3% de esas ordenes tambien llevo [Lemon Sparkling Water]
  -> eso es 75.6x mas probable que si fuera puro azar (lift=75.63)
  -> esta combinacion aparece en el 0.10% de TODAS las ordenes (support=0.0010)



## 7. Versión legible con nombres de producto

Guardamos también una copia de las reglas con los nombres de producto en texto plano, para poder leerlas de un vistazo sin ir a buscar cada `product_id`.

In [10]:
rules_readable = rules.copy()
rules_readable['antecedents_names'] = rules_readable['antecedents'].apply(
    lambda s: [id_to_name.get(p, f'?{p}') for p in s]
)
rules_readable['consequents_names'] = rules_readable['consequents'].apply(
    lambda s: [id_to_name.get(p, f'?{p}') for p in s]
)
rules_readable[['antecedents_names', 'consequents_names', 'support', 'confidence', 'lift']].head(50)

,antecedents_names,consequents_names,support,confidence,lift
0,[Non Fat Acai & Mixed Berries Yogurt],[Icelandic Style Skyr Blueberry Non-fat Yogurt],0.001221,0.453129,75.647888
1,[Icelandic Style Skyr Blueberry Non-fat Yogurt],[Non Fat Acai & Mixed Berries Yogurt],0.001221,0.203822,75.647888
2,[Grapefruit Sparkling Water],[Lemon Sparkling Water],0.001037,0.223406,75.634068
3,[Lemon Sparkling Water],[Grapefruit Sparkling Water],0.001037,0.351201,75.634068
4,[Icelandic Style Skyr Blueberry Non-fat Yogurt],[Non Fat Raspberry Yogurt],0.002247,0.375136,73.640837
5,[Non Fat Raspberry Yogurt],[Icelandic Style Skyr Blueberry Non-fat Yogurt],0.002247,0.441106,73.640837
6,[Total 2% Lowfat Greek Strained Yogurt With Bl...,[Total 2% with Strawberry Lowfat Greek Straine...,0.001161,0.179834,72.141821
7,[Total 2% with Strawberry Lowfat Greek Straine...,[Total 2% Lowfat Greek Strained Yogurt With Bl...,0.001161,0.465810,72.141821
8,[Nonfat Icelandic Style Strawberry Yogurt],[Icelandic Style Skyr Blueberry Non-fat Yogurt],0.001418,0.427553,71.378186
9,[Icelandic Style Skyr Blueberry Non-fat Yogurt],[Nonfat Icelandic Style Strawberry Yogurt],0.001418,0.236745,71.378186


## 7.5 Política de recomendación: no recomendar lo obvio

Si mirás la tabla de reglas de arriba, vas a notar que **banana** (`24852`, 14,7% de las órdenes) y **bag of organic bananas** (`13176`, 11,8%) aparecen por todos lados: casi la mitad de las 1.803 reglas las involucran. Vale la pena entender bien por qué pasa esto antes de "arreglarlo".

**No es un bug — es la firma de un producto ultra-popular.** La banana la compra casi todo el mundo, así que co-ocurre con casi cualquier otra cosa con **confidence** alta. Pero si medimos el **lift** de esas reglas (la métrica que corrige por popularidad), la mediana es apenas **1,74** — contra **2,76** de las reglas que no tocan banana. Es decir: las reglas de banana son, en promedio, las *más débiles* de toda la tabla. El lift ya nos está avisando "che, esto es popularidad pura, no una relación real" — el problema es que con `MIN_LIFT=1.0` igual las dejamos pasar.

**Decisión de diseño:** en vez de tocar la tabla `rules` (la dejamos intacta, con las 1.803 reglas, para no perder información de análisis), aplicamos la política **"no recomendar lo trivial"** en la capa de recomendación: `recommend_from_cart` va a descartar cualquier producto candidato cuyo soporte global supere `MAX_CONSEQUENT_SUPPORT`. Ningún cliente necesita que le digamos que compre banana — la iba a comprar igual.

**Seamos honestos sobre qué es esto en la práctica: es un blocklist, construido automáticamente desde los datos.** Con `MAX_CONSEQUENT_SUPPORT=0.10`, los únicos productos que superan ese soporte son exactamente las dos bananas — así que a este umbral, el filtro es matemáticamente idéntico a poner `blocklist = {24852, 13176}` a mano. La ventaja frente a un blocklist manual es que se recalcula solo si cambia el dataset, y generaliza a **cualquier** producto ultra-popular, no solo a la banana. Si bajás el umbral, empieza a excluir más productos populares (frutillas, palta, espinaca...) — lo vemos en la celda siguiente.

**El costo real, sin maquillarlo:** este filtro también descarta señal legítima. Por ejemplo, en el carrito `espinaca orgánica + palta orgánica`, la banana orgánica aparecía con **lift 2,96** — eso no es puro ruido de popularidad, es la firma real del "carrito de comprador orgánico". Ese tipo de recomendación se pierde con este filtro. Es un trade-off consciente: preferimos nunca recomendar lo obvio, a costa de perder alguna asociación banana genuina de vez en cuando.

In [11]:
# reusamos el item_support ya calculado en la seccion 4 (soporte global de CADA producto,
# no solo los que sobrevivieron al filtro de mineria)
product_support = pd.Series(item_support, index=all_product_ids)

blocked = product_support[product_support > MAX_CONSEQUENT_SUPPORT].sort_values(ascending=False)
print(f'con MAX_CONSEQUENT_SUPPORT={MAX_CONSEQUENT_SUPPORT}, se excluyen {len(blocked)} productos como recomendacion:')
for pid, supp in blocked.items():
    print(f'  {pid:>6}  {supp*100:5.1f}%  {id_to_name[pid]}')

con MAX_CONSEQUENT_SUPPORT=0.1, se excluyen 2 productos como recomendacion:
   24852   14.7%  Banana
   13176   11.8%  Bag of Organic Bananas


## 8. Función recomendadora: de un carrito a una lista de sugerencias

`recommend_from_cart` recibe una lista de `product_id` (el carrito actual) y devuelve productos sugeridos. La lógica es:

1. Nos quedamos solo con las reglas cuyo `antecedents` sea un **subconjunto** del carrito (es decir, todos los productos que la regla necesita ya están en el carrito).
2. Descartamos de las sugerencias los productos que **ya están** en el carrito (no tiene sentido recomendar algo que el cliente ya eligió).
3. Descartamos los productos cuyo soporte global supera `MAX_CONSEQUENT_SUPPORT` (la política "no recomendar lo obvio" de la sección 7.5).
4. Si varias reglas distintas sugieren el mismo producto, nos quedamos con la de mejor `lift` (y `confidence` como desempate).
5. Ordenamos todo por `lift` descendente y devolvemos el top-N.

In [12]:
def recommend_from_cart(cart_product_ids, rules_df, top_n=10, max_consequent_support=MAX_CONSEQUENT_SUPPORT):
    """Recomienda productos a partir de un carrito, usando las reglas de asociacion.

    Parameters
    ----------
    cart_product_ids : list[int]
        product_id de los productos que ya estan en el carrito.
    rules_df : pd.DataFrame
        reglas con columnas ['antecedents', 'consequents', 'support', 'confidence', 'lift'].
    top_n : int
        cantidad maxima de recomendaciones a devolver.
    max_consequent_support : float or None
        no se recomienda ningun producto cuyo soporte global (popularidad) supere este valor
        (ver seccion 7.5: evita recomendar productos obvios como banana). Pasa None para
        desactivar este filtro y ver el comportamiento sin de-sesgar.
    """
    cart = set(cart_product_ids)

    # regla aplicable = todos sus antecedentes ya estan en el carrito
    applicable = rules_df[rules_df['antecedents'].apply(lambda a: a.issubset(cart))]

    best_per_product = {}
    for _, r in applicable.iterrows():
        for pid in r['consequents']:
            if pid in cart:
                continue  # no recomendamos algo que ya esta en el carrito
            if max_consequent_support is not None and product_support.get(pid, 0) > max_consequent_support:
                continue  # producto demasiado popular: no aporta valor recomendarlo (seccion 7.5)
            candidate = (r['lift'], r['confidence'])
            if pid not in best_per_product or candidate > best_per_product[pid][:2]:
                best_per_product[pid] = (r['lift'], r['confidence'], pid)

    if not best_per_product:
        return pd.DataFrame(columns=['product_id', 'product_name', 'confidence', 'lift'])

    recs = pd.DataFrame(
        [{'product_id': pid, 'lift': lift, 'confidence': conf} for lift, conf, pid in best_per_product.values()]
    )
    recs['product_name'] = recs['product_id'].map(id_to_name)
    recs = recs.sort_values(['lift', 'confidence'], ascending=False).head(top_n).reset_index(drop=True)
    return recs[['product_id', 'product_name', 'confidence', 'lift']]

## 9. Prueba manual: armamos un carrito y vemos qué recomienda

Vamos a simular un carrito con dos productos típicos de compra de productos frescos: **espinaca orgánica** y **palta orgánica**. Podés cambiar los `product_id` de la lista `cart` por cualquier otro (podés buscarlos en `products.csv` filtrando por `product_name`) y volver a correr la celda para ver cómo cambian las recomendaciones.

In [13]:
cart = [21903, 47209]  # Organic Baby Spinach, Organic Hass Avocado
print('Carrito actual:')
for pid in cart:
    print(f'  - {pid}: {id_to_name[pid]}')

print()
print(f'Recomendaciones SIN de-sesgar (max_consequent_support=None):')
display(recommend_from_cart(cart, rules, max_consequent_support=None))

print()
print(f'Recomendaciones CON de-sesgar (max_consequent_support={MAX_CONSEQUENT_SUPPORT}, el default):')
recommend_from_cart(cart, rules)

Carrito actual:
  - 21903: Organic Baby Spinach
  - 47209: Organic Hass Avocado

Recomendaciones SIN de-sesgar (max_consequent_support=None):


,product_id,product_name,confidence,lift
0,30391,Organic Cucumber,0.120369,4.813555
1,5876,Organic Lemon,0.126615,4.638983
2,22935,Organic Yellow Onion,0.130684,3.704022
3,24964,Organic Garlic,0.119796,3.508253
4,45007,Organic Zucchini,0.114037,3.497459
5,27966,Organic Raspberries,0.137732,3.230718
6,13176,Bag of Organic Bananas,0.349446,2.960663
7,21137,Organic Strawberries,0.233174,2.832160
8,26209,Limes,0.116530,2.663984
9,47766,Organic Avocado,0.127682,2.321534



Recomendaciones CON de-sesgar (max_consequent_support=0.1, el default):


,product_id,product_name,confidence,lift
0,30391,Organic Cucumber,0.120369,4.813555
1,5876,Organic Lemon,0.126615,4.638983
2,22935,Organic Yellow Onion,0.130684,3.704022
3,24964,Organic Garlic,0.119796,3.508253
4,45007,Organic Zucchini,0.114037,3.497459
5,27966,Organic Raspberries,0.137732,3.230718
6,21137,Organic Strawberries,0.233174,2.832160
7,26209,Limes,0.116530,2.663984
8,47766,Organic Avocado,0.127682,2.321534


Comparando ambas tablas: **`Bag of Organic Bananas` (13176) aparece en la versión sin de-sesgar y desaparece en la versión con de-sesgar** — quedó excluida por superar `MAX_CONSEQUENT_SUPPORT`, aunque tenía un lift decente (2,96). Ese es el trade-off explicado en la sección 7.5, ahora en acción sobre un caso real.

Probá con otro carrito para ver cómo cambia la recomendación (por ejemplo, buscá algún `product_id` en `products.csv`):

In [14]:
# products[products['product_name'].str.contains('Strawberr', case=False)]

cart_2 = [20114, 30391]  # Organic Whole Strawberries
print('Carrito actual:')
for pid in cart_2:
    print(f'  - {pid}: {id_to_name[pid]}')

print()
print('Recomendaciones:')
recommend_from_cart(cart_2, rules)

Carrito actual:
  - 20114: Jalapeno Peppers
  - 30391: Organic Cucumber

Recomendaciones:


,product_id,product_name,confidence,lift
0,28842,Bunched Cilantro,0.135288,9.552700
1,31717,Organic Cilantro,0.137215,6.344980
2,26209,Limes,0.267686,6.119579
3,44359,Organic Small Bunch Celery,0.106739,5.039638
4,5876,Organic Lemon,0.103331,3.785890
5,24964,Organic Garlic,0.123352,3.612402
6,47209,Organic Hass Avocado,0.217136,3.268339
7,45007,Organic Zucchini,0.101863,3.124103
8,47626,Large Lemon,0.147717,3.110849
9,22935,Organic Yellow Onion,0.106553,3.020069


**¿Qué pasa si el carrito no tiene ninguna regla aplicable?** Por ejemplo, si armamos un carrito con un `product_id` que quedó fuera del filtro de soporte mínimo (un producto poco comprado), `recommend_from_cart` va a devolver una tabla **vacía** — no es un error, es el modelo diciendo honestamente "no tengo evidencia suficiente para recomendar nada acá" (esto es el efecto de *cold-start* que mencionamos en las conclusiones).

## 10. Guardar las reglas (opcional)

Guardamos las reglas calculadas en `data/processed/` para poder reutilizarlas sin tener que recalcular todo el pipeline de nuevo.

In [15]:
rules.to_csv(OUTPUT_PATH, index=False)
print(f'reglas guardadas en {OUTPUT_PATH}')

reglas guardadas en ../data/processed/association_rules.csv


## 11. Exportar el modelo para uso fuera del notebook

Hasta acá todo lo que hicimos vive **adentro de este notebook**. Para usarlo desde otro código (un script, una API, otro proyecto) hace falta "exportarlo": empaquetar todo lo que `recommend_from_cart` necesita en un solo archivo autocontenido, y dar una forma prolija de importarlo.

**Por qué no alcanza con leer los CSV del otro lado:** `data/processed/association_rules.csv` y `data/raw/instacart/products.csv` están en `.gitignore` — no viajan con el repositorio. Si el módulo exportado dependiera de leerlos en runtime, dejaría de ser portable apenas salís de esta máquina. La solución es empaquetar todo (reglas + soporte global + nombres de producto) en **un único artefacto binario** con `joblib` (ya está en `requirements.txt`, no hace falta instalar nada).

**Dónde vive el código reusable:** [`src/recommender.py`](../src/recommender.py), clase `Recommender`. Tiene la **misma lógica** que `recommend_from_cart` de la sección 8 — la duplicación es intencional: acá arriba la función queda para leerla paso a paso mientras aprendés; `src/recommender.py` es la versión que se importa desde otro lado. A diferencia de `recommend_from_cart`, devuelve una lista de dicts `{'product_id', 'product_name', 'rank'}` — sin exponer `lift`/`confidence`, el orden ya queda reflejado en `rank` (1 = mejor recomendación).

In [16]:
import joblib
from datetime import datetime, timezone

MODEL_PATH = '../models/mba_prod_recommender.joblib'

artifact = {
    'rules': rules,
    'product_support': product_support,
    'id_to_name': id_to_name,
    'max_consequent_support': MAX_CONSEQUENT_SUPPORT,
    'metadata': {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'min_support': MIN_SUPPORT,
        'min_confidence': MIN_CONFIDENCE,
        'min_lift': MIN_LIFT,
        'n_rules': len(rules),
        'source_notebook': '06_Market_Basket_Analysis.ipynb',
    },
}

joblib.dump(artifact, MODEL_PATH)
print(f'modelo exportado a {MODEL_PATH}')
print(f"reglas empaquetadas: {artifact['metadata']['n_rules']:,}")

modelo exportado a ../models/mba_prod_recommender.joblib
reglas empaquetadas: 1,803


### Probando que el modelo exportado funciona standalone

Importamos `Recommender` desde `src/` (afuera del notebook, como lo haría cualquier otro script) y comparamos, para los mismos dos carritos que ya usamos en la sección 9, que el **orden de `product_id`** coincide exactamente con `recommend_from_cart`.

In [17]:
import sys
sys.path.append('../src')
from mba_prod_recommender import MBA_Prod_Recommender

model = MBA_Prod_Recommender.load()  # sin argumentos: encuentra models/recommender.joblib solo
print('modelo cargado, metadata:', model.metadata)

modelo cargado, metadata: {'generated_at': '2026-07-02T22:12:52.076440+00:00', 'min_support': 0.001, 'min_confidence': 0.1, 'min_lift': 1.0, 'n_rules': 1803, 'source_notebook': '06_Market_Basket_Analysis.ipynb'}


In [18]:
for test_cart in [cart, cart_2]:
    ids_notebook = recommend_from_cart(test_cart, rules)['product_id'].tolist()
    ids_module = [r['product_id'] for r in model.recommend(test_cart)]

    print('carrito:', [id_to_name[p] for p in test_cart])
    print('  recommend_from_cart :', ids_notebook)
    print('  MBA_Prod_Recommender.recommend:', ids_module)
    print('  mismo orden de product_id:', ids_notebook == ids_module)
    print()

print('salida de MBA_Prod_Recommender.recommend (formato final, exportable):')
model.recommend(cart)

carrito: ['Organic Baby Spinach', 'Organic Hass Avocado']
  recommend_from_cart : [30391, 5876, 22935, 24964, 45007, 27966, 21137, 26209, 47766]
  MBA_Prod_Recommender.recommend: [30391, 5876, 22935, 24964, 45007, 27966, 21137, 26209, 47766]
  mismo orden de product_id: True

carrito: ['Jalapeno Peppers', 'Organic Cucumber']
  recommend_from_cart : [28842, 31717, 26209, 44359, 5876, 24964, 47209, 45007, 47626, 22935]
  MBA_Prod_Recommender.recommend: [28842, 31717, 26209, 44359, 5876, 24964, 47209, 45007, 47626, 22935]
  mismo orden de product_id: True

salida de MBA_Prod_Recommender.recommend (formato final, exportable):


[{'product_id': 30391, 'product_name': 'Organic Cucumber', 'rank': 1},
 {'product_id': 5876, 'product_name': 'Organic Lemon', 'rank': 2},
 {'product_id': 22935, 'product_name': 'Organic Yellow Onion', 'rank': 3},
 {'product_id': 24964, 'product_name': 'Organic Garlic', 'rank': 4},
 {'product_id': 45007, 'product_name': 'Organic Zucchini', 'rank': 5},
 {'product_id': 27966, 'product_name': 'Organic Raspberries', 'rank': 6},
 {'product_id': 21137, 'product_name': 'Organic Strawberries', 'rank': 7},
 {'product_id': 26209, 'product_name': 'Limes', 'rank': 8},
 {'product_id': 47766, 'product_name': 'Organic Avocado', 'rank': 9}]

## Conclusiones

**Cómo leer las métricas:**
- **support** = qué tan frecuente es la combinación en el total de órdenes (popularidad).
- **confidence** = de las órdenes con el antecedente, qué porcentaje también tiene el consecuente (fiabilidad de la regla).
- **lift** = cuánto más probable es la combinación comparado con el azar puro. `lift > 1` es lo que buscamos para recomendar; más alto es mejor.

**Por qué FP-Growth:** construye un árbol compacto (FP-tree) en vez de generar candidatos exhaustivamente como Apriori, así que escala mucho mejor con miles de productos y millones de órdenes — el resultado matemático (los itemsets/reglas) es idéntico al que daría Apriori con el mismo `min_support`.

**Cómo validamos el modelo en este notebook (validación "simple"):**
1. Las métricas intrínsecas de las reglas (support/confidence/lift) — nos dicen si un patrón es frecuente y estadísticamente significativo dentro de los datos con los que se entrenó.
2. La prueba manual armando un carrito y revisando si las recomendaciones tienen sentido de negocio (¿son productos que uno esperaría ver juntos?).

**Limitaciones a tener en cuenta:**
- Esto **no mide qué tan bien predice sobre compras futuras no vistas** — solo mide qué tan fuertes son los patrones dentro del propio historial que usamos para entrenar. Es fácil confundir "la regla es estadísticamente fuerte" con "el modelo predice bien", y no es lo mismo.
- Hay un efecto de **cold-start**: un producto cuyo soporte individual está por debajo de `MIN_SUPPORT` nunca va a aparecer en ninguna regla, así que un carrito armado solo con productos poco comunes no va a recibir recomendaciones (podés comprobarlo armando un carrito con un `product_id` de un producto raro).

**Si en algún momento querés ir un paso más allá:** la forma correcta de medir qué tan bien recomienda el modelo sobre datos no vistos es una **evaluación offline con hold-out**: entrenar las reglas con `order_products__prior.csv` (como hicimos acá), y para cada orden de `order_products__train.csv`, esconder un producto, armar el carrito con el resto, y chequear si el modelo lo recomienda entre sus top-N sugerencias (esto se mide con métricas como **hit-rate** o **precision@k / recall@k**). No lo hicimos en este notebook por decisión explícita de mantenerlo simple, pero queda como el próximo paso natural.